# Support Vector Regressor (SVR)
### Insurance Cost Prediction — Regression Track

## 1. Introduction
This notebook implements and evaluates a **Support Vector Regressor** for predicting medical insurance
charges from demographic and health-related attributes. The preprocessing pipeline (encoding,
train/test split, and scaling) is kept identical to the other regression notebooks in this
project so that all algorithms are compared on the same test set.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Fixed seed for reproducible results
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
sns.set_palette('colorblind')


## 2. Data Loading
The dataset (`insurance.csv`) contains `age`, `sex`, `bmi`, `children`, `smoker`, `region`, and
the target variable `charges`.


In [ ]:
df = pd.read_csv('insurance.csv')
print("Dataset shape:", df.shape)
df.head()


In [ ]:
# Check data types and missing values before preprocessing
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())


## 3. Data Preprocessing
Categorical variables are one-hot encoded, the data is split into training and test sets
(80:20), and numeric features are scaled. The scaler is fit on the training set only to avoid
data leakage.


In [ ]:
# One-hot encode categorical variables
df_enc = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

X = df_enc.drop('charges', axis=1)
y = df_enc['charges']

# 80:20 train/test split, fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Scale features using parameters learned from the training set only
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print("Training set:", X_train.shape, " Test set:", X_test.shape)


## 4. Model Training and Hyperparameter Tuning
SVR is distance-based, so the scaled features are used for both training and evaluation.


In [ ]:
from sklearn.svm import SVR

# Baseline model with default parameters
svr_baseline = SVR()
svr_baseline.fit(X_train_scaled, y_train)
y_pred_base = svr_baseline.predict(X_test_scaled)

print("Baseline R2  :", round(r2_score(y_test, y_pred_base), 4))
print("Baseline RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_base)), 2))
print("Baseline MAE :", round(mean_absolute_error(y_test, y_pred_base), 2))


In [ ]:
# Tune C, kernel, gamma, and epsilon using 5-fold cross-validation
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.01, 0.1, 0.5],
}

grid_search = GridSearchCV(
    SVR(),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)
grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV R2 :", round(grid_search.best_score_, 4))

best_svr = grid_search.best_estimator_


In [ ]:
# Compare tuned model against the baseline
y_pred_tuned = best_svr.predict(X_test_scaled)
print("Tuned R2  :", round(r2_score(y_test, y_pred_tuned), 4), " (baseline:", round(r2_score(y_test, y_pred_base), 4), ")")
print("Tuned RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_tuned)), 2), " (baseline:", round(np.sqrt(mean_squared_error(y_test, y_pred_base)), 2), ")")


## 5. Model Evaluation
The tuned model is evaluated on the held-out test set using R2, RMSE, and MAE, and validated
with 5-fold cross-validation for stability.


In [ ]:
y_pred = best_svr.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score : {r2:.4f}")
print(f"RMSE     : {rmse:.2f}")
print(f"MAE      : {mae:.2f}")

results_svr = {'Model': 'Support Vector Regressor', 'R2': r2, 'RMSE': rmse, 'MAE': mae}
results_svr


In [ ]:
# 5-fold cross-validation to check consistency of R2 across different splits
cv_scores = cross_val_score(best_svr, X_train_scaled, y_train, cv=5, scoring='r2')
print("5-fold CV R2 scores:", np.round(cv_scores, 4))
print("Mean CV R2         :", round(cv_scores.mean(), 4))
print("Std  CV R2         :", round(cv_scores.std(), 4))


## 6. Visualization and Analysis

In [ ]:
# Predicted vs actual charges; points closer to the diagonal indicate better fit
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5, edgecolor='k')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Ideal fit')
plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title('Support Vector Regressor: Predicted vs Actual Charges')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Residuals should be randomly scattered around zero with no clear pattern
residuals = y_test - y_pred

plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.5, edgecolor='k')
plt.axhline(0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Charges')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Support Vector Regressor: Residual Plot')
plt.tight_layout()
plt.show()


Support Vector Regressor has no built-in feature importance, so **permutation importance**
is used instead — it measures the drop in R2 when a feature's values are randomly shuffled.
The model was trained on scaled features, so the scaled test set is used here as well.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_svr, X_test_scaled, y_test, n_repeats=10,
                               random_state=RANDOM_STATE, scoring='r2')
perm_importances = pd.Series(perm.importances_mean, index=X_test_scaled.columns).sort_values()

plt.figure(figsize=(7, 5))
perm_importances.plot(kind='barh', color=sns.color_palette('colorblind'))
plt.xlabel('Mean Decrease in R2')
plt.title('Support Vector Regressor: Permutation Feature Importance')
plt.tight_layout()
plt.show()


## 7. Conclusion
- The tuned **Support Vector Regressor** was evaluated on the held-out test set using R2, RMSE, and MAE
  (reported in the Model Evaluation section above).
- 5-fold cross-validation confirms the stability of this result across different data splits.
- The predicted-vs-actual and residual plots indicate how well the model generalizes to unseen
  data, and the feature-importance plot highlights which attributes most influence the
  predicted insurance charges.
